# 06 (v2) - Release validation on CPU: weights check, real-weight regression, HTTP check

**What this notebook does.** It reruns the Phase 2 pipeline with the evaluated weights on the private 401-photo team set (CPU), then runs a local HTTP check of the service. It does **not** train, does not change any threshold and does not overwrite earlier results.

**What the results are.** A regression check on photographs that were already inspected during development. It is **not** a new untouched test and **not** a T4 benchmark; CPU timings must not be compared with the T4 figures in the report.

**Reference results** (produced on 31 October 2026 by the previous version of this notebook, source `051bd2c`):

| Check | W&B run | Recorded values |
|---|---|---|
| Real-weight CPU regression | [f8manl0i](https://wandb.ai/yasser-alhajlah-kaust/smart-pet-vision/runs/f8manl0i) | 401 photos, 386/399 single-species correct, 81 referrals, 17 multi-animal ready, guards on all |
| Local HTTP check (CPU, 1 worker) | [mu2tolkg](https://wandb.ai/yasser-alhajlah-kaust/smart-pet-vision/runs/mu2tolkg) | 300/300 HTTP 200, guards 300/300, p50/p95/p99 383.8/430.8/690.1 ms, 415 for text, 400 for empty |

**Changes from the previous version.** Paths are set in one cell; a preflight step checks files, weight hashes, operating point and versions before anything runs; every attempt (including failures) is appended to `attempts.jsonl`; the HTTP step selects demonstration images only from this run's real `predictions.csv` and stops if that file is missing (the previous version could write placeholder files instead); W&B logging is optional, separates the two runs by job type and records commit, environment, weight hashes and data protocol.

**Run time (CPU runtime):** about 10-15 minutes for 401 photos plus about 3 minutes for the HTTP check. **Secrets:** the W&B key is read from Colab Secrets (`WANDB_API_KEY`) or a hidden prompt; never paste it into a cell.

Run the cells in order from a **new** runtime. After the run: File -> Save a copy in Drive keeps the notebook with its outputs.


In [ ]:
# 1. Configuration - the only cell you should need to edit
REV = "051bd2cff8c362aabdca82b3a9189c54275368cc"   # source snapshot validated in the report
REPO_URL = "https://github.com/Yasser-20-alhajlah/smart-pet-vision"
DRIVE_PROJECT = "/content/drive/MyDrive/capstone /smart-pet-vision"          # holds runs/manifest.json and weights
TEAM_DATA = "/content/drive/MyDrive/capstone /SmartPetVision_TeamData"       # holds images/ and labels.csv
OUTPUT_BASE = "/content/drive/MyDrive/capstone /Submission/FINAL PHASE_2/03_Evidence"  # a new timestamped folder is created inside
RUN_HTTP = True
HTTP_REQUESTS = 300
LOG_TO_WANDB = False          # set True only after the local results look right
WANDB_ENTITY, WANDB_PROJECT = "yasser-alhajlah-kaust", "smart-pet-vision"
WANDB_HOST = None             # e.g. "https://api.forge.coreweave.com" if the team account uses that host


In [ ]:
# 2. Setup: Drive, pinned source, pinned dependencies
import os, sys, json, time, subprocess, hashlib, platform
from pathlib import Path
from google.colab import drive
drive.mount("/content/drive")
STAMP = time.strftime("%Y%m%dT%H%M%SZ", time.gmtime())
WORK = Path("/content") / f"spv_{STAMP}"
subprocess.run(["git", "clone", "--quiet", REPO_URL, str(WORK)], check=True)
subprocess.run(["git", "-C", str(WORK), "checkout", "--quiet", "--detach", REV], check=True)
head = subprocess.run(["git", "-C", str(WORK), "rev-parse", "HEAD"], capture_output=True, text=True, check=True).stdout.strip()
assert head == REV, f"Checked out {head}, expected {REV}"
REPO = WORK / "smart-pet-vision" / "phase2"
REQ = REPO.parent / "requirements.txt"
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", str(REQ)], check=True)
# Preflight and evidence tools come from the hardening branch; they do not change model code.
TOOLS_REV = "phase2-submission-hardening"
subprocess.run(["git", "-C", str(WORK), "fetch", "--quiet", "origin", TOOLS_REV], check=True)
subprocess.run(["git", "-C", str(WORK), "checkout", "FETCH_HEAD", "--", "smart-pet-vision/phase2/tools/preflight_release.py"], check=True)
sys.path[:0] = [str(REPO / "src"), str(REPO / "tools")]
IMAGES, LABELS = Path(TEAM_DATA) / "images", Path(TEAM_DATA) / "labels.csv"
OUT = Path(OUTPUT_BASE) / f"release_validation_{STAMP}"
ATTEMPTS = Path(OUTPUT_BASE) / "attempts.jsonl"
def record_attempt(step, status, **extra):
    ATTEMPTS.parent.mkdir(parents=True, exist_ok=True)
    with ATTEMPTS.open("a") as f:
        f.write(json.dumps(dict(stamp=STAMP, step=step, status=status, source_revision=REV, **extra)) + "\n")
print("Source", head, "| output", OUT)


In [ ]:
# 3. Preflight: stop early with a clear message if anything is missing or different
import preflight_release as pf
manifest_path = Path(DRIVE_PROJECT) / "runs" / "manifest.json"
try:
    PRE = pf.run(manifest_path, images=IMAGES, labels=LABELS, output=OUT, requirements=REQ)
except pf.PreflightError as e:
    record_attempt("preflight", "failed", error=str(e))
    raise
if PRE["version_mismatches"]:
    print("Installed versions differ from the pinned file (recorded, not fatal):", *PRE["version_mismatches"], sep="\n  ")
record_attempt("preflight", "passed", n_images=PRE["n_images"], weights_sha256=PRE["weights_sha256"])
print(json.dumps(PRE, indent=2))


In [ ]:
# 4. Real-weight CPU regression on the full cohort (no subset, no training)
cmd = [sys.executable, str(REPO / "tools" / "validate_release.py"), "--manifest", str(manifest_path),
       "--images", str(IMAGES), "--labels", str(LABELS), "--output", str(OUT), "--threads", "4"]
LOG = WORK / "validation_execution.log"
with LOG.open("w") as log:
    res = subprocess.run(cmd, cwd=str(REPO), stdout=log, stderr=subprocess.STDOUT, text=True)
print(LOG.read_text()[-4000:])
if res.returncode != 0:
    record_attempt("validation", "failed", returncode=res.returncode, log_tail=LOG.read_text()[-500:])
    raise RuntimeError(f"Validation failed (exit {res.returncode}); see {LOG}. Not marked as passed.")
import pandas as pd
SUMMARY = json.loads((OUT / "summary.json").read_text())
PRED = pd.read_csv(OUT / "predictions.csv")
if len(PRED) != PRE["n_images"]:
    record_attempt("validation", "failed", error="predictions row count differs from cohort")
    raise RuntimeError(f"predictions.csv has {len(PRED)} rows for {PRE['n_images']} images")
import shutil
shutil.copy2(LOG, OUT / "execution.log")
PROV = dict(source_revision=REV, weights_sha256=PRE["weights_sha256"], label_sha256=pf.sha256(LABELS),
            cohort="private team set (supplementary regression, not a new test)", n_images=PRE["n_images"],
            device="CPU", python=platform.python_version(), fresh_independent_test=False, returncode=0)
(OUT / "validation_provenance.json").write_text(json.dumps(PROV, indent=2))
record_attempt("validation", "passed", n=SUMMARY["n"], accuracy=SUMMARY.get("accuracy"), referrals=SUMMARY["referrals"])
print(json.dumps(SUMMARY, indent=2))


In [ ]:
# 5. Local HTTP check on CPU (one worker). Timing uses only the first HTTP_REQUESTS sorted photos.
import requests, numpy as np, base64
from IPython.display import display
from PIL import Image
from io import BytesIO
if not RUN_HTTP:
    raise SystemExit("RUN_HTTP is False; HTTP check skipped (state this when reporting).")
images = sorted(IMAGES.glob("*.jpg"))
if len(images) < HTTP_REQUESTS:
    raise RuntimeError(f"Need {HTTP_REQUESTS} images, found {len(images)}; the protocol is not reduced silently.")
HTTP_OUT = OUT / f"http_{time.strftime('%Y%m%dT%H%M%SZ', time.gmtime())}"
HTTP_OUT.mkdir()
env = {**os.environ, "PYTHONPATH": str(REPO / "src"), "SPV_MANIFEST": str(manifest_path), "CUDA_VISIBLE_DEVICES": ""}
url = "http://127.0.0.1:8765"
server_log = (HTTP_OUT / "server.log").open("w")
proc = subprocess.Popen([sys.executable, "-m", "uvicorn", "spv.serve:app", "--host", "127.0.0.1", "--port", "8765", "--workers", "1"],
                        cwd=str(REPO), env=env, stdout=server_log, stderr=subprocess.STDOUT)
def post(path, with_images=False):
    with path.open("rb") as f:
        return requests.post(url + "/analyze", params={"images": str(with_images).lower()},
                             files={"file": (path.name, f, "image/jpeg")}, timeout=120)
try:
    t0 = time.perf_counter(); health = None
    while time.perf_counter() - t0 < 300 and health is None:
        if proc.poll() is not None: raise RuntimeError("Service exited during startup; see server.log")
        try:
            r = requests.get(url + "/health", timeout=5)
            if r.ok and r.json().get("status") == "ok": health = r.json()
        except (requests.RequestException, ValueError): time.sleep(1)
    if health is None: raise RuntimeError("Service did not become healthy within 300 s; see server.log")
    startup_s = time.perf_counter() - t0
    for p in images[:5]: post(p)                                    # warm-up, not timed
    lat, codes, ok, guards = [], {}, 0, 0
    for i, p in enumerate(images[:HTTP_REQUESTS]):
        t = time.perf_counter(); r = post(p); lat.append((time.perf_counter() - t) * 1000)
        codes[str(r.status_code)] = codes.get(str(r.status_code), 0) + 1
        body = r.json() if r.status_code == 200 else {}
        ok += int(r.status_code == 200 and "verdict" in body)
        guards += int(body.get("staff_approval_required") is True and body.get("automatic_publication_allowed") is False)
    bad = requests.post(url + "/analyze", files={"file": ("x.txt", b"x", "text/plain")}, timeout=60).status_code
    empty = requests.post(url + "/analyze", files={"file": ("empty.jpg", b"", "image/jpeg")}, timeout=60).status_code
    # Demonstration images: chosen ONLY from this run's real predictions (one cat, one dog, one referral).
    demo = []
    for subset in (PRED[(PRED.true == "cat") & (PRED.verdict == "ready")], PRED[(PRED.true == "dog") & (PRED.verdict == "ready")], PRED[PRED.verdict != "ready"]):
        if subset.empty: raise RuntimeError("No real prediction available for a demo category; not substituting placeholders.")
        demo.append(IMAGES / f"{subset.image.iloc[0]}.jpg")
    for i, p in enumerate(demo, 1):
        r = post(p, True); r.raise_for_status(); body = r.json()
        print("Demo", i, p.name, body.get("species"), body.get("verdict"), [x.get("code") for x in body.get("reasons", [])])
        if body.get("annotated_png"):
            data = base64.b64decode(body["annotated_png"]); (HTTP_OUT / f"demo_{i}_annotated.png").write_bytes(data)
            display(Image.open(BytesIO(data)))
finally:
    proc.terminate()
    try: proc.wait(timeout=30)
    except subprocess.TimeoutExpired: proc.kill(); proc.wait()
    server_log.close()
p50, p95, p99 = (float(x) for x in np.percentile(lat, [50, 95, 99]))
HTTP = dict(device="CPU", protocol=f"local uvicorn, one worker, /analyze?images=false; first {HTTP_REQUESTS} sorted team JPEGs",
            warmup=5, requests=HTTP_REQUESTS, succeeded=ok, success_rate=ok / HTTP_REQUESTS,
            publication_guards_passed_requests=guards, status_codes=codes, health=health, startup_s=startup_s,
            http_latency_ms=dict(p50=p50, p95=p95, p99=p99), non_image_status=bad, empty_upload_status=empty,
            source_revision=REV, weights_sha256=PRE["weights_sha256"],
            passed_service_check=(ok >= 0.99 * HTTP_REQUESTS and guards == HTTP_REQUESTS and bad == 415 and empty == 400),
            independent_new_test=False, demo_images=[p.name for p in demo], timing_includes_demo=False)
(HTTP_OUT / "http_summary.json").write_text(json.dumps(HTTP, indent=2))
record_attempt("http", "passed" if HTTP["passed_service_check"] else "failed", p95_ms=p95, succeeded=ok, guards=guards)
print(json.dumps(HTTP, indent=2))
if not HTTP["passed_service_check"]:
    raise RuntimeError("HTTP check did not meet its own criteria; results saved but not marked as passed.")


In [ ]:
# 6. Compare with the reference runs (f8manl0i, mu2tolkg). Differences are reported, not hidden.
REF = {"n": 401, "classification_n": 399, "accuracy": 386 / 399, "referrals": 81, "multi_animal_ready": 17}
for k, v in REF.items():
    got = SUMMARY.get(k)
    print(f"{k:22s} reference {v!s:22s} this run {got!s:22s} {'same' if got is not None and abs(float(got) - float(v)) < 1e-9 else 'DIFFERENT'}")
if RUN_HTTP:
    print("HTTP p50/p95/p99 ms this run:", *(round(HTTP['http_latency_ms'][k], 1) for k in ('p50', 'p95', 'p99')),
          "| reference 383.8 / 430.8 / 690.1 (CPU timings vary by machine; not comparable with T4)")


In [ ]:
# 7. Optional: log to W&B as NEW runs (only results produced above; earlier results are not re-uploaded)
if LOG_TO_WANDB:
    import wandb
    try:
        from google.colab import userdata
        key = userdata.get("WANDB_API_KEY")
    except Exception:
        from getpass import getpass
        key = getpass("W&B API key (hidden): ")
    wandb.login(key=key, host=WANDB_HOST, relogin=True, verify=True); del key
    env_rec = SUMMARY.get("environment", {})
    common = dict(source_revision=REV, weights_sha256=PRE["weights_sha256"], device="CPU", environment=env_rec,
                  data_protocol=dict(cohort="private team set, previously inspected", split="supplementary-regression",
                                     n_images=PRE["n_images"], label_sha256=PROV["label_sha256"], oxford_test_split=False),
                  fresh_independent_test=False, notebook="06_release_validation_v2")
    with wandb.init(entity=WANDB_ENTITY, project=WANDB_PROJECT, name=f"cpu-regression-{STAMP}", job_type="inference-validation",
                    group=f"release-validation-{STAMP}", tags=["phase2", "cpu", "regression", "real-weights"], config=common) as run:
        run.summary.update({k: v for k, v in SUMMARY.items() if isinstance(v, (int, float, bool, str))})
        art = wandb.Artifact("phase2-cpu-regression", type="validation", description="CPU regression on previously inspected team photos (predictions only, no images).")
        for name in ("summary.json", "predictions.csv", "validation_provenance.json", "execution.log"): art.add_file(str(OUT / name))
        run.log_artifact(art); val_url = run.url
    print("Validation run:", val_url)
    if RUN_HTTP:
        with wandb.init(entity=WANDB_ENTITY, project=WANDB_PROJECT, name=f"cpu-http-{STAMP}", job_type="inference-http",
                        group=f"release-validation-{STAMP}", tags=["phase2", "cpu", "http"], config=common) as run:
            run.summary.update({k: v for k, v in HTTP.items() if isinstance(v, (int, float, bool, str))})
            run.summary.update({f"http_latency_{k}_ms": v for k, v in HTTP["http_latency_ms"].items()})
            art = wandb.Artifact("phase2-cpu-http", type="validation", description="Aggregate HTTP results only; photographs and annotated images stay in private Drive.")
            art.add_file(str(HTTP_OUT / "http_summary.json")); run.log_artifact(art); http_url = run.url
        print("HTTP run:", http_url)
    record_attempt("wandb", "logged")
else:
    print("W&B logging skipped (LOG_TO_WANDB = False). Results are in", OUT)
